## Filter NACC data into cases with all corrct answers, all incorrect answers and the ones in between

Created by: Sahana Kowshik

Date: 05/21/2025

In [1]:
import pandas as pd

In [2]:
train_data = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/training_data_grpo/train_summary.csv")
data = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/adrd_simplified_evaluation/llm_answer_extractor/extracted_results/qwen3b_Train.csv")

In [3]:
len(data)

356328

In [4]:
def modify(results_df):
    results_combined = results_df.copy()[['prediction','ground_truth', 'ID']]
    results_combined['prediction'] = results_combined['prediction']#.replace({'Y': 'A', 'N': 'B'})
    # results_combined['ground_truth'] = results_combined['ground_truth']#.replace({'Yes': 'A', 'No': 'B'})
    results_combined['correctness'] = results_combined['prediction'] == results_combined['ground_truth']
    # p = len(results_combined) // n
    # results_combined['run'] = list(range(1,p+1))*n
    results_combined = results_combined.reset_index(names=['problem']).reset_index(drop=True)
    return results_combined

In [5]:
modified_data = modify(data)

In [7]:
summary_df = modified_data.groupby('ID').agg(
    group_size=('correctness', 'size'),
    num_correct=('correctness', 'sum')  # True counts as 1
).reset_index()

In [8]:
all_correct = summary_df[summary_df['num_correct'] == 8].reset_index(drop=True)
all_incorrect = summary_df[(summary_df['num_correct'] == 0) & (summary_df['group_size'] == 8)].reset_index(drop=True)
invalid_df = summary_df[summary_df['group_size'] != 8].reset_index(drop=True)

In [9]:
assert len(invalid_df) == 0

In [10]:
remaining = summary_df[
    (~summary_df['ID'].isin(all_correct['ID'])) &
    (~summary_df['ID'].isin(all_incorrect['ID'])) &
    (~summary_df['ID'].isin(invalid_df['ID']))
].reset_index(drop=True)


In [11]:
remaining_data = train_data[train_data['ID'].isin(remaining['ID'])].reset_index(drop=True)
all_correct_data = train_data[train_data['ID'].isin(all_correct['ID'])].reset_index(drop=True)
all_incorrect_data = train_data[train_data['ID'].isin(all_incorrect['ID'])].reset_index(drop=True)

In [12]:
# print(f"Number of invalid cases: {len(invalid_data)}")
print(f"Number of all correct cases: {len(all_correct_data)}")
print(f"Number of all incorrect cases: {len(all_incorrect_data)}")
print(f"Number of selected cases: {len(remaining_data)}")

Number of all correct cases: 3832
Number of all incorrect cases: 9496
Number of selected cases: 31213


In [13]:
assert len(remaining_data) + len(all_correct_data) + len(all_incorrect_data) == len(train_data)

In [14]:
len(train_data)

44541

In [19]:
all_correct_data['Q_TYPE'].value_counts()

Q_TYPE
Primary etiology    2492
Cognitive status    1119
Amyloid PET          170
Neuropath             46
DATSCAN                4
MCI subtype            1
Name: count, dtype: int64

In [14]:
remaining_data.to_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/select_train_cases/train_selected.csv", index=False)
all_correct_data.to_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/select_train_cases/all_correct.csv", index=False)
all_incorrect_data.to_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/select_train_cases/all_incorrect.csv", index=False)

In [16]:
pet_keys_remove = [
    'Abnormally elevated amyloid on PET',
    'Abnormally low amyloid in CSF',
    'FDG-PET pattern of AD',
    'Tau PET evidence for AD',
    'Abnormally elevated CSF Tau or pTau'
]

In [17]:
for key in pet_keys_remove:
    for i, row in remaining_data[remaining_data['Q_TYPE'] == "Amyloid PET"].iterrows():
        if key in row['patient_summary']:
            print(i, key)
            # break

In [18]:
dat_keys_remove = [
    'Dopamine transporter scan (DATscan) evidence for Lewy body disease'
]

In [19]:
for key in dat_keys_remove:
    for i, row in remaining_data[remaining_data['Q_TYPE'] == "DATSCAN"].iterrows():
        if key in row['patient_summary']:
            print(i, key)
            # break

In [23]:
x = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc/training_data/testing_data_grpo/test_summary.csv")

/scratch/5552643.1.cds-gpu-long/ipykernel_1930111/2109210281.py:1: DtypeWarning: Columns (20,22,24,28,43,45,50,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,155,164,175,178,216,219,221,223,225,227,229,231,233,235,237,239,241,243,245,247,249,251,253,255,257,259,261,263,265,267,269,271,398,400,420,422,431,444,453,571,599,607,663,679,696,699,716,727,733,808,819,825,892,948,949,957,958,959,960,998,1017,1022,1192,1196,1199,1395,1397,1399,1400,1402,1409,1411,1413,1414,1421,1423,1425,1427,1428,1435,1450,1464,1478,1492,1494,1530,1546,1548,1550,1552,1554,1556,1558,1560,1562,1564,1566,1568,1570,1572,1574,1576,1578,1580,1582,1584,1586,1588,1590,1592,1594,1596,1598,1600,1650,1651,1653,1654,1657,1658,1661,1662,1665,1666,1669,1670,1744,1803,1812,1814,1816,1818,1829,1831,1833,1841,1843,1845,1847,1855,1857,1859,1861,1887) have mixed types. Specify dtype option on import or set low_memory=False.
  x = pd.read_csv("/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-mo

In [25]:
set(x['NACCID']).intersection(set(train_data['ID']))

set()